In [ ]:
from transformers import AutoTokenizer
tokenizer_base = AutoTokenizer.from_pretrained("meta-llama/Llama-3.1-8B")
tokenizer_base.save_pretrained("Llama-3.1-8B-Base")


In [ ]:
import os, glob
import pickle as pkl
import json
from collections import defaultdict
os.makedirs("./VocabFiles/CPT-SupremeCourtCases/Llama-3.1_Tokenizers/",exist_ok=True)

for fname in glob.glob("./VocabFiles/CPT-SupremeCourtCases/CPT-SupremeCourtCases-Llama3.1_Vocab/*"):
    print('***********Processing:',fname)
    vocab_to_merges = defaultdict(list)
    
    tokenizer_base = AutoTokenizer.from_pretrained("meta-llama/Llama-3.1-8B")
    
    tokenizer_json = json.load(open("Llama-3.1-8B-Base/tokenizer.json"))
    vocab_base = tokenizer_json["model"]["vocab"]
    merges_base = tokenizer_json["model"]["merges"]
    
    words_to_add = open(fname,'r',encoding='utf-8').read().splitlines()
    words_to_add = sorted(words_to_add, key=lambda x: len(x))

    for word in words_to_add:
        if 'Ĕ' in word: continue
        if 'Ä' in word: continue
        if 'ą' in word: continue
        
        split = tokenizer_base.tokenize(word if not word.startswith('Ġ') else ' '+word[1:])
        
        flag = False
        for sp in split:
            if 'Ä' in sp: 
                flag = True
                break
        
        if flag: continue
        
        if len(split) == 1:
            continue
        
        if len(split) == 2: #pass
            if word not in vocab_to_merges:
                vocab_to_merges[word] = [split[0],split[1]]
        
        if len(split) >= 3:
            # print('--word:',word,split)
            new_word = split[0]
            for i in range(1,len(split)):
                left = new_word
                right = split[i]
                new_word += split[i]
                # print('new_word:',new_word, 'Merge:',left,right)
                if new_word not in vocab_to_merges:
                    vocab_to_merges[new_word] = [left,right]
    
    idx = 0
    for key,val in vocab_to_merges.items():
        if key not in tokenizer_json["model"]["vocab"]:
            # print(key,val,idx,tokenizer_base.tokenize(key if not key.startswith('Ġ') else ' '+key[1:]))
            tokenizer_json["model"]["vocab"][key] = 128000+idx
            tokenizer_json["model"]["merges"].append(val)
            idx += 1
        
    tokenizer_json['post_processor']['processors'][-1]['special_tokens']['<|begin_of_text|>']['ids'] = [128000+idx]
    
    dump_dir = f'./VocabFiles/CPT-SupremeCourtCases/Llama-3.1_Tokenizers/SupremeCourtCases_MEDVOC-LLM_Llama3.1_{fname.split("/")[-1][:-4]}'
    tokenizer_base.save_pretrained(dump_dir)
    
    with open(dump_dir+'/tokenizer.json', 'w',encoding='utf-8') as f:
        json.dump(tokenizer_json, f)
    f.close()
    
    tokenizer_dump = AutoTokenizer.from_pretrained(dump_dir)
    tokenizer_dump.save_pretrained(dump_dir)


In [ ]:
import pandas as pd
df = pd.read_csv('../../data/Legal_Val_OOV_Tokens.csv')

freq_ebm = df['Count'].to_list()
terms_EBM = df['Word'].to_list()
split_bart = df['Splits'].to_list()

sum_num = 0.
sum_den = 0.
for idx,term in enumerate(terms_EBM):
    sum_num += split_bart[idx]*freq_ebm[idx]
    sum_den += freq_ebm[idx]

old_score = sum_num/sum_den


import glob
from tqdm import tqdm
from collections import defaultdict
from transformers import AutoTokenizer

tokenizer_base = AutoTokenizer.from_pretrained("meta-llama/Llama-3.1-8B")

dict_scores = defaultdict(lambda : defaultdict(dict))
for fname in tqdm(sorted(glob.glob('./VocabFiles/CPT-SupremeCourtCases/Llama-3.1_Tokenizers/*'), key=lambda x: float(x.split('/')[-1].split('_')[-2][:-1]))): 
    # print('***********Processing:',fname, fname.split('/')[-1].split('_')[1])
    try:
        domain_tok = AutoTokenizer.from_pretrained(fname)
        
        sum_num = 0.
        sum_den = 0.
        
        for idx,term in enumerate(terms_EBM):
            sum_num += len(domain_tok.tokenize(term))*freq_ebm[idx]
            sum_den += freq_ebm[idx]

        dict_scores[fname.split('/')[-1].split('_')[-2]] = [sum_num/sum_den,len(domain_tok)-len(tokenizer_base)]
    except Exception as e:
        print('Error:', e,fname)
        continue


with open(f'CPT-Eval-SupremeCourtCases-FERTILITY','a') as f:
    f.write(f'\n-------------\nCPT-Llama3.1 []\n--------------------\n')
    f.write('BASE_Tok: '+str(round(old_score,2))+'\n')
    
    for key,val in dict_scores.items():
        f.write(f'{key} : {val[0]} || {val[1]}\n')


In [ ]:
x = list(dict_scores.keys())
y = list(dict_scores.values())
y = [a[0] for a in y]
import matplotlib.pyplot as plt

plt.plot(x, y)
plt.xticks(rotation=90)
plt.grid()


In [1]:
import pandas as pd
df_train_oov = pd.read_csv('../../data/Legal_Train_OOV_Tokens_Cleaned.csv')

In [2]:
from transformers import AutoTokenizer

tokenizer_base = AutoTokenizer.from_pretrained("meta-llama/Llama-3.1-8B")
tokenizer_10K = AutoTokenizer.from_pretrained('./VocabFiles/CPT-SupremeCourtCases/Llama-3.1_Tokenizers/SupremeCourtCases_MEDVOC-LLM_Llama3.1_10.0_')


In [3]:
added_vocab = set(tokenizer_10K.get_vocab().keys()) - set(tokenizer_base.get_vocab().keys())
print('Added Vocab Size:', len(added_vocab))

added_vocab_list = list(added_vocab)
added_vocab_list = sorted(added_vocab_list, key=lambda x: len(x))


Added Vocab Size: 11397


In [4]:
df_train_oov.describe(percentiles=[0.25,0.5,0.75,0.9,0.95,0.99])
df_train_oov = df_train_oov[df_train_oov['Count'] >= 13]


In [5]:
import tqdm

potential_whole_words = []

def is_match(word, token):
    if token.startswith('Ġ'):
        token = token[1:]
    
    if word.strip() == token.strip():
        return True

    return False 

for word in tqdm.tqdm(added_vocab_list, desc='Finding Matches', total=len(added_vocab_list)):
    mask = df_train_oov['Word'].apply(lambda x: is_match(x, word))

    if not df_train_oov[mask].empty:
        potential_whole_words.append([word, df_train_oov[mask]['Count'].sum()])


Finding Matches: 100%|██████████| 11397/11397 [01:08<00:00, 167.06it/s]


In [6]:
potential_whole_words = sorted(potential_whole_words, key=lambda x: x[1], reverse=True)
potential_whole_words = [x[0].replace('Ġ','') for x in potential_whole_words]

potential_whole_words = list(set(potential_whole_words))
potential_whole_words = sorted(potential_whole_words, key=lambda x: len(x))

In [7]:
with open('./VocabFiles/CPT-SupremeCourtCases/CPT-SupremeCourtCases-Llama3.1_Vocab/Llama-3.1-8B-PotentialWholeWords.txt', 'w', encoding='utf-8') as f:
    for item in potential_whole_words:
        f.write(item.strip()+'\n')
        
        

In [9]:
potential_whole_words

['Pw',
 'xv',
 'KI',
 'Sd',
 'yd',
 'Ld',
 'CJ',
 'Ss',
 'Ws',
 'aJ',
 'DEY',
 'LAL',
 'GOI',
 'OSA',
 'FIR',
 'BOC',
 'xii',
 'gms',
 'SMT',
 'vii',
 'SGL',
 'Raj',
 'Bom',
 'LOK',
 'FSI',
 'NOC',
 'SEB',
 'GOs',
 'DCR',
 'PSC',
 'SLP',
 'Sri',
 'VRS',
 'CJI',
 'DAS',
 'Nai',
 'Jus',
 'Mfg',
 'RAJ',
 'RAY',
 'AER',
 'Rau',
 'LAO',
 'GCM',
 'bly',
 'UCO',
 'Pvt',
 'MCI',
 'anr',
 'DRT',
 'SBI',
 'RTI',
 'edn',
 'LDC',
 'Tis',
 'ANR',
 'POs',
 'RAO',
 'HAD',
 'Anr',
 'HHC',
 'AHs',
 'EIA',
 'IBC',
 'Adi',
 'Crl',
 'Lok',
 'LPG',
 'AAI',
 'KAR',
 'MOU',
 'MoU',
 'SIR',
 'Wad',
 'ILR',
 'wad',
 'Bux',
 'MPT',
 'RIL',
 'CAs',
 'BOB',
 'TDS',
 'NTT',
 'TWI',
 'DTC',
 'RPA',
 'AOR',
 'HAI',
 'deh',
 'ITO',
 'CRP',
 'Asa',
 'CJM',
 'PSU',
 'UGC',
 'PLP',
 'Tej',
 'NTC',
 'SIT',
 'SRI',
 'Rly',
 'EPC',
 'ICI',
 'DCI',
 'Ors',
 'IAS',
 'STs',
 'CCL',
 'Kha',
 'Exh',
 'ATC',
 'KLT',
 'ors',
 'Diu',
 'HSR',
 'HON',
 'TPC',
 'PWD',
 'RFA',
 'Etc',
 'bai',
 'XXV',
 'crl',
 'TMC',
 'CVC',
 'Guj',
 '

In [10]:
from transformers import AutoTokenizer
tokenizer_base = AutoTokenizer.from_pretrained("meta-llama/Llama-3.1-8B")
tokenizer_base.add_tokens(potential_whole_words)

6249

In [11]:
tokenizer_base.save_pretrained('./VocabFiles/CPT-SupremeCourtCases/Llama-3.1_Tokenizers/SupremeCourtCases_MEDVOC-LLM_Llama3.1_10.0_WholeWords/')

('./VocabFiles/CPT-SupremeCourtCases/Llama-3.1_Tokenizers/SupremeCourtCases_MEDVOC-LLM_Llama3.1_10.0_WholeWords/tokenizer_config.json',
 './VocabFiles/CPT-SupremeCourtCases/Llama-3.1_Tokenizers/SupremeCourtCases_MEDVOC-LLM_Llama3.1_10.0_WholeWords/special_tokens_map.json',
 './VocabFiles/CPT-SupremeCourtCases/Llama-3.1_Tokenizers/SupremeCourtCases_MEDVOC-LLM_Llama3.1_10.0_WholeWords/tokenizer.json')